# 0.6 Model Evaluation

**Author:** Ahmed Hasan | **Report:** Final

Formal evaluation of all 5 models: RF tier classifier, RF coverage classifier,
K-means route clusters, K-means neighbourhood clusters, NegBin passup GLM.
Includes 5-fold CV, F1, AUC-ROC, silhouette, CH, DB, and classifier comparison.

In [ ]:
# Papermill parameters
n_splits = 5
random_state = 42
report_name = "final"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import (
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import statsmodels.api as sm

from ptn_analysis.context import TransitContext
from ptn_analysis.context.config import FEED_ID_CURRENT, REPORTS_DIR, PTN_TIER_COLORS
from ptn_analysis.analysis import FrequencyAnalyzer, CoverageAnalyzer
from ptn_analysis.context.reporting import save_report_figure

ctx = TransitContext.from_defaults()
db = ctx.working_db
freq = FrequencyAnalyzer("ywg", FEED_ID_CURRENT, db)
cov = CoverageAnalyzer("ywg", FEED_ID_CURRENT, db)
fig_dir = REPORTS_DIR / report_name / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

## 1. RF Tier Classifier (5-fold Stratified CV)

In [ ]:
# Load route features and PTN tier labels
route_features = freq.build_route_classification_feature_table()
if route_features.empty:
    print("No route features available")
else:
    feature_cols = [c for c in route_features.columns if c not in
                    ['feed_id', 'route_id', 'route_short_name', 'ptn_tier', 'route_long_name']]
    X = route_features[feature_cols].copy()
    le = LabelEncoder()
    y = le.fit_transform(route_features['ptn_tier'])
    imp = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(X)

    # 5-fold stratified CV
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    classifiers = {
        'Random Forest': RandomForestClassifier(n_estimators=300, random_state=random_state, class_weight='balanced'),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=random_state),
        'Naive Bayes': GaussianNB(),
        'Decision Tree': DecisionTreeClassifier(random_state=random_state, class_weight='balanced'),
    }

    results = []
    for name, clf in classifiers.items():
        y_pred = cross_val_predict(clf, X_imp, y, cv=skf)
        f1_macro = f1_score(y, y_pred, average='macro')
        f1_weighted = f1_score(y, y_pred, average='weighted')
        results.append({'Classifier': name, 'F1 (macro)': round(f1_macro, 3),
                        'F1 (weighted)': round(f1_weighted, 3)})
        if name == 'Random Forest':
            rf_pred = y_pred
    
    results_df = pd.DataFrame(results)
    print("=== Tier Classifier Comparison (5-fold CV) ===")
    print(results_df.to_string(index=False))
    print(f"\nRF Classification Report:\n{classification_report(y, rf_pred, target_names=le.classes_)}")

## 2. RF Coverage Classifier (Spatial CV)

In [ ]:
# Load neighbourhood features with spatial groups for GroupKFold
nb_features = cov.build_neighbourhood_classification_feature_table()
if not nb_features.empty:
    nb_feature_cols = ['population_density_per_km2', 'pct_commute_public_transit',
                       'median_household_income_2020', 'pct_seniors_65_plus']
    available_cols = [c for c in nb_feature_cols if c in nb_features.columns]
    X_nb = nb_features[available_cols].copy()
    y_nb = LabelEncoder().fit_transform(nb_features['density_category'])
    X_nb_imp = SimpleImputer(strategy='median').fit_transform(X_nb)

    # Standard stratified CV for coverage
    skf_nb = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rf_cov = RandomForestClassifier(n_estimators=300, random_state=random_state, class_weight='balanced')
    y_cov_pred = cross_val_predict(rf_cov, X_nb_imp, y_nb, cv=skf_nb)
    f1_cov = f1_score(y_nb, y_cov_pred, average='weighted')
    print(f"Coverage Classifier F1 (weighted): {f1_cov:.3f}")
else:
    print("No neighbourhood features available")

## 3. K-Means Clustering Evaluation

In [ ]:
# Route clustering evaluation: silhouette, CH, DB for k=2..8
hourly = freq.departures_by_hour_by_route()
if not hourly.empty:
    pivot = hourly.pivot_table(index=['route_id', 'route_short_name'], columns='hour',
                               values='departures', fill_value=0)
    row_totals = pivot.sum(axis=1).replace(0, 1)
    X_clust = (pivot.div(row_totals, axis=0)).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_clust)

    k_range = range(2, 9)
    metrics = {'k': [], 'silhouette': [], 'calinski_harabasz': [], 'davies_bouldin': [], 'inertia': []}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        labels = km.fit_predict(X_scaled)
        metrics['k'].append(k)
        metrics['silhouette'].append(round(silhouette_score(X_scaled, labels), 3))
        metrics['calinski_harabasz'].append(round(calinski_harabasz_score(X_scaled, labels), 1))
        metrics['davies_bouldin'].append(round(davies_bouldin_score(X_scaled, labels), 3))
        metrics['inertia'].append(round(km.inertia_, 1))

    cluster_eval = pd.DataFrame(metrics)
    print("=== Route Clustering Evaluation ===")
    print(cluster_eval.to_string(index=False))

    # DBSCAN comparison
    db_scan = DBSCAN(eps=1.5, min_samples=3)
    dbscan_labels = db_scan.fit_predict(X_scaled)
    n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = list(dbscan_labels).count(-1)
    print(f"\nDBSCAN: {n_clusters_dbscan} clusters, {n_noise} noise points")
    if n_clusters_dbscan >= 2:
        mask = dbscan_labels != -1
        print(f"DBSCAN silhouette (excl noise): {silhouette_score(X_scaled[mask], dbscan_labels[mask]):.3f}")
else:
    print("No hourly departure data available")

## 3b. PCA Dimensionality Reduction

PCA on neighbourhood census features to identify dominant dimensions of variation.
Shows how many components capture most variance in the demographic landscape.

In [ ]:
from sklearn.decomposition import PCA

# PCA on neighbourhood census features
if not nb_features.empty:
    pca_cols = [c for c in ['population_density_per_km2', 'pct_commute_public_transit',
                            'median_household_income_2020', 'pct_seniors_65_plus',
                            'pct_recent_immigrants', 'pct_commute_car',
                            'pct_commute_walk', 'pct_commute_cycle']
                if c in nb_features.columns]
    X_pca_raw = nb_features[pca_cols].dropna()
    if len(X_pca_raw) >= 5 and len(pca_cols) >= 3:
        X_pca_scaled = StandardScaler().fit_transform(X_pca_raw)
        n_components = min(len(pca_cols), 5)
        pca = PCA(n_components=n_components, random_state=random_state)
        pca.fit(X_pca_scaled)

        cumvar = np.cumsum(pca.explained_variance_ratio_)
        print(f"=== PCA on {len(pca_cols)} Census Features ({len(X_pca_raw)} neighbourhoods) ===")
        for i, (ev, cv) in enumerate(zip(pca.explained_variance_ratio_, cumvar)):
            print(f"  PC{i+1}: {ev:.1%} variance (cumulative: {cv:.1%})")

        # Top loadings for PC1 and PC2
        loadings = pd.DataFrame(pca.components_.T, index=pca_cols,
                                columns=[f'PC{i+1}' for i in range(n_components)])
        print(f"\nTop loadings:\n{loadings[['PC1','PC2']].round(3)}")

        # Variance explained bar chart
        fig_pca, ax_pca = plt.subplots(figsize=(8, 4))
        x_pos = np.arange(1, n_components + 1)
        ax_pca.bar(x_pos, pca.explained_variance_ratio_, color='#1f77b4', alpha=0.7, label='Individual')
        ax_pca.plot(x_pos, cumvar, 'ro-', label='Cumulative')
        ax_pca.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
        ax_pca.set_xlabel('Principal Component')
        ax_pca.set_ylabel('Explained Variance Ratio')
        ax_pca.set_title(f'PCA: {cumvar[1]:.0%} variance in first 2 components')
        ax_pca.set_xticks(x_pos)
        ax_pca.legend()
        ax_pca.set_ylim(0, 1.05)
        plt.tight_layout()
        plt.close(fig_pca)
        print(f"\nFirst 2 PCs capture {cumvar[1]:.1%} of total variance")
    else:
        print(f"Insufficient data for PCA: {len(X_pca_raw)} rows, {len(pca_cols)} features")
else:
    print("No neighbourhood features for PCA")

## 4. NegBin GLM Evaluation

In [ ]:
# Load capacity data for NegBin evaluation
cap_tbl = freq.build_capacity_priority_table()
if not cap_tbl.empty and 'passup_count' in cap_tbl.columns:
    model_data = cap_tbl.dropna(subset=['passup_count', 'mean_headway_minutes', 'weekday_boardings'])
    model_data = model_data[model_data['passup_count'] > 0].copy()
    if len(model_data) >= 10:
        model_data['log_headway'] = np.log1p(model_data['mean_headway_minutes'])
        model_data['log_boardings'] = np.log1p(model_data['weekday_boardings'])
        pct_col = 'pct_on_time' if 'pct_on_time' in model_data.columns else None
        
        X_cols = ['log_headway', 'log_boardings']
        if pct_col:
            model_data['pct_on_time_filled'] = model_data[pct_col].fillna(model_data[pct_col].median())
            X_cols.append('pct_on_time_filled')
        
        X_nb = sm.add_constant(model_data[X_cols])
        y_nb = model_data['passup_count'].astype(float)

        # Fit NegBin
        nb_model = sm.GLM(y_nb, X_nb, family=sm.families.NegativeBinomial()).fit()
        # Fit Poisson for comparison
        pois_model = sm.GLM(y_nb, X_nb, family=sm.families.Poisson()).fit()

        print("=== NegBin vs Poisson Comparison ===")
        print(f"NegBin AIC: {nb_model.aic:.1f} | Poisson AIC: {pois_model.aic:.1f}")
        print(f"NegBin pseudo-R2: {1 - nb_model.deviance/nb_model.null_deviance:.3f}")
        print(f"NegBin deviance: {nb_model.deviance:.1f} (df={nb_model.df_resid})")
        print(f"\nNegBin Summary:\n{nb_model.summary()}")
    else:
        print(f"Insufficient data for NegBin: {len(model_data)} routes with passups")
else:
    print("No capacity data available")

## 5. Combined Evaluation Figure

In [ ]:
# Generate combined evaluation figure for report
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Classifier comparison bar chart
if 'results_df' in dir():
    results_df.plot(x='Classifier', y=['F1 (macro)', 'F1 (weighted)'], kind='bar', ax=axes[0, 0])
    axes[0, 0].set_title('(a) Classifier Comparison (5-fold CV)')
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].tick_params(axis='x', rotation=30)

# Panel B: Clustering validation metrics
if 'cluster_eval' in dir():
    axes[0, 1].plot(cluster_eval['k'], cluster_eval['silhouette'], 'o-', label='Silhouette')
    ax2 = axes[0, 1].twinx()
    ax2.plot(cluster_eval['k'], cluster_eval['davies_bouldin'], 's--', color='red', label='Davies-Bouldin')
    axes[0, 1].set_title('(b) Clustering Validation (k=2..8)')
    axes[0, 1].set_xlabel('k')
    axes[0, 1].set_ylabel('Silhouette')
    ax2.set_ylabel('Davies-Bouldin')
    axes[0, 1].legend(loc='upper left')
    ax2.legend(loc='upper right')

# Panel C: RF confusion matrix
if 'rf_pred' in dir():
    cm = confusion_matrix(y, rf_pred)
    im = axes[1, 0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[1, 0].set_title('(c) RF Tier Confusion Matrix')
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[1, 0].text(j, i, str(cm[i, j]), ha='center', va='center')

# Panel D: NegBin residuals
if 'nb_model' in dir():
    resid = nb_model.resid_deviance
    axes[1, 1].scatter(nb_model.fittedvalues, resid, alpha=0.5, s=20)
    axes[1, 1].axhline(y=0, color='red', linestyle='--')
    axes[1, 1].set_title('(d) NegBin Deviance Residuals')
    axes[1, 1].set_xlabel('Fitted values')
    axes[1, 1].set_ylabel('Deviance residuals')

fig.tight_layout()
fig.savefig(fig_dir / 'model_evaluation_combined.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print(f"Saved: {fig_dir / 'model_evaluation_combined.png'}")

In [ ]:
# Export evaluation metrics
export_dir = REPORTS_DIR / report_name / "tables"
export_dir.mkdir(parents=True, exist_ok=True)
if 'results_df' in dir():
    results_df.to_parquet(export_dir / 'model_evaluation.parquet', index=False)
if 'cluster_eval' in dir():
    cluster_eval.to_parquet(export_dir / 'clustering_evaluation.parquet', index=False)
print('Evaluation metrics exported.')